In [4]:
import pandas as pd
import holidays 
import duckdb

In [5]:
country_codes = [
    "ET",  # Ethiopia
    "ER",  # Eritrea
    "SV",  # El Salvador
    "CN",  # China
    "IN",  # India
    "PK",  # Pakistan
    "BD",  # Bangladesh
    "VN",  # Vietnam
    "US",  # United States
    "CA"   # Canada
]

country_code_replacements = {
                    "ET":"Ethiopia",
                    "ER":"Eritrea",
                    "SV":"El Salvador",
                    "CN":"China",
                    "IN":"India",
                    "PK":"Pakistan",
                    "BD":"Bangladesh",
                    "VN":"Vietnam",
                    "US":"United  States",
                    "CA":"Canada"
                }

In [ ]:
con = duckdb.connect("/opt/airflow/data/Processed/bikeshare.duckdb")

IOException: IO Error: Cannot open file "C:\Users\Darshil\Documents\Code\CP_BIKESHARE\CPB_AIRFLOW\dags\Data\Processed\database.duckdb": The system cannot find the path specified.


In [ ]:
dim_date = con.execute("SELECT DISTINCT(Start_date) AS Date FROM Silver;").df()

In [ ]:
all_holidays = []

for code in country_codes:
    h = holidays.CountryHoliday(code, years=2020)
    for date, name in h.items():
        all_holidays.append({"country": code, "holiday_date": date, "holiday_name": name})
holiday_list = pd.DataFrame(all_holidays)

C:\Users\Darshil\AppData\Local\Temp\ipykernel_24472\3209154753.py:4: DeprecationWarning: CountryHoliday is deprecated, use country_holidays instead.
  h = holidays.CountryHoliday(code, years=2020)


In [ ]:
holiday_list["holiday_date"] = pd.to_datetime(holiday_list["holiday_date"])

In [ ]:
holiday_list["country"] = holiday_list["country"].replace(country_code_replacements)

In [ ]:
countries = holiday_list.groupby("holiday_date")["country"].apply(list).reset_index()
holidays = holiday_list.groupby("holiday_date")["holiday_name"].apply(list).reset_index()

In [ ]:
final = pd.merge(countries, holidays, on = "holiday_date")
final["holiday_count"] = final["country"].apply(len)
final["is_holiday"] = 1

In [ ]:
final = pd.merge(left = dim_date, right = final, how = "left", left_on = "Date", right_on = "holiday_date")

In [ ]:
final

,Date,holiday_date,country,holiday_name,holiday_count,is_holiday
0,2020-01-09,NaT,NaN,NaN,NaN,NaN
1,2020-01-31,2020-01-31,[China],[春节延长假期],1.0,1.0
2,2020-01-20,2020-01-20,"[Ethiopia, Eritrea, United States]","[የጥምቀት በዓል, Epiphany, Martin Luther King Jr. Day]",3.0,1.0
3,2020-01-25,2020-01-25,"[China, Vietnam]","[春节, Tết Nguyên Đán]",2.0,1.0
4,2020-01-26,2020-01-26,"[China, India, Vietnam]","[春节, Republic Day, Mùng hai Tết Nguyên Đán]",3.0,1.0
5,2020-01-12,NaT,NaN,NaN,NaN,NaN
6,2020-01-22,NaT,NaN,NaN,NaN,NaN
7,2020-01-29,2020-01-29,"[China, Vietnam]","[春节（补假）, Mùng năm Tết Nguyên Đán]",2.0,1.0
8,2020-01-05,NaT,NaN,NaN,NaN,NaN
9,2020-01-15,NaT,NaN,NaN,NaN,NaN


In [ ]:
final = final[["Date", "country", "holiday_name", "holiday_count", "is_holiday"]]

In [ ]:
final["day"] = final["Date"].dt.day_name()

In [ ]:
long_weekends = final.loc[final["day"].isin(["Friday", "Monday"])].index
final["long_weekend"] = 0
final.loc[long_weekends, "long_weekend"] = 1

In [ ]:
longer_weekends = final.loc[final["day"].isin(["Tuesday", "Thursday"])].index
final["mini_vac_blocker"] = 0
final.loc[longer_weekends, "mini_vac_blocker"] = 1

In [ ]:
final["is_holiday"] = final["is_holiday"].fillna(0, inplace = True)
final["holiday_count"] = final["holiday_count"].fillna(0, inplace = True)

C:\Users\Darshil\AppData\Local\Temp\ipykernel_24472\1220374192.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  final["is_holiday"] = final["is_holiday"].fillna(0, inplace = True)
C:\Users\Darshil\AppData\Local\Temp\ipykernel_24472\1220374192.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series thr

In [ ]:
final["holiday_count"] = final["holiday_count"].astype(int)
final["is_holiday"] = final["is_holiday"].astype(int)

In [ ]:
final.loc[final["day"].isin(["Saturday", "Sunday"]), "is_holiday"] = 1

In [ ]:
final["country"] = final["country"].fillna("none")
final["holiday_name"] = final["holiday_name"].fillna("none")

In [ ]:
final.columns = ["Date", "Country", "Holiday", "Holiday_count", "Is_holiday", "Day", "Long_weekend", "Mini_vac_blocker"]
final = final[["Date", "Day", "Is_holiday", "Holiday_count", "Long_weekend", "Mini_vac_blocker", "Holiday", "Country"]]

In [ ]:
final

,Date,Day,Is_holiday,Holiday_count,Long_weekend,Mini_vac_blocker,Holiday,Country
0,2020-01-09,Thursday,0,0,0,1,none,none
1,2020-01-31,Friday,1,1,1,0,[春节延长假期],[China]
2,2020-01-20,Monday,1,3,1,0,"[የጥምቀት በዓል, Epiphany, Martin Luther King Jr. Day]","[Ethiopia, Eritrea, United States]"
3,2020-01-25,Saturday,1,2,0,0,"[春节, Tết Nguyên Đán]","[China, Vietnam]"
4,2020-01-26,Sunday,1,3,0,0,"[春节, Republic Day, Mùng hai Tết Nguyên Đán]","[China, India, Vietnam]"
5,2020-01-12,Sunday,1,0,0,0,none,none
6,2020-01-22,Wednesday,0,0,0,0,none,none
7,2020-01-29,Wednesday,1,2,0,0,"[春节（补假）, Mùng năm Tết Nguyên Đán]","[China, Vietnam]"
8,2020-01-05,Sunday,1,0,0,0,none,none
9,2020-01-15,Wednesday,0,0,0,0,none,none


In [ ]:
con.execute("""

CREATE SCHEMA IF NOT EXISTS GOLD;

CREATE TABLE GOLD.DIM_DATE AS SELECT * FROM final;

""").df()

,Count
0,31
